In [ ]:
!pip install -q -U google-generativeai pydantic

import json
from datetime import datetime
from pydantic import BaseModel
from google.colab import userdata
import google.generativeai as genai

# Setup API Key
api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)

# Local Validation Schema
class InvoiceData(BaseModel):
    invoice_number: str
    supplier_name: str
    base_amount: float
    gst_amount: float
    total_amount: float

class LedgerLinkAgent:
    def __init__(self):
        # Using the exact model version your API key requires
        self.model = genai.GenerativeModel('gemini-3.8-flash')
        self.compliance_rules = {
            "eway_bill_threshold": 50000.0,
            "standard_gst_rate": 0.18
        }

    def _role_extractor(self, raw_text: str) -> InvoiceData:
        prompt = f"""
        Extract the invoice details from the following raw text.
        Raw Text: "{raw_text}"

        Return ONLY a raw JSON object. Do not include markdown formatting or backticks.
        Keys must be exactly: "invoice_number", "supplier_name", "base_amount", "gst_amount", "total_amount".
        """

        # The most basic, stable API call possible (no config to cause hangs)
        response = self.model.generate_content(prompt)

        # Strip away any accidental markdown backticks the AI might add
        clean_json = response.text.strip().removeprefix('```json').removesuffix('```').strip()

        # Parse and validate
        data = json.loads(clean_json)
        return InvoiceData(**data)

    def _role_verifier(self, invoice: InvoiceData) -> list[str]:
        flags = []
        if invoice.base_amount < 0 or invoice.total_amount < 0:
            flags.append("ERROR: Negative monetary values detected.")

        if invoice.total_amount > self.compliance_rules["eway_bill_threshold"]:
            flags.append("URGENT: Total exceeds ₹50,000. E-way bill generation required.")

        expected_gst = round(invoice.base_amount * self.compliance_rules["standard_gst_rate"], 2)
        if round(invoice.gst_amount, 2) != expected_gst:
            flags.append(f"WARNING: GST mismatch. Expected ₹{expected_gst}, got ₹{invoice.gst_amount}.")

        return flags

    def prepare_accountant_handoff(self, raw_text: str):
        print("--- 1. Agent Extracting Messy Input ---")
        try:
            invoice = self._role_extractor(raw_text)
        except Exception as e:
            print(f"Error during extraction: {e}")
            return None

        print("--- 2. Python Engine Verifying Compliance Guardrails ---")
        flags = self._role_verifier(invoice)

        report = {
            "status": "Ready for Accountant Review",
            "date_processed": datetime.today().strftime('%Y-%m-%d'),
            "structured_ledger_entry": invoice.model_dump(),
            "compliance_flags": flags if flags else ["All checks passed successfully."]
        }

        print("\n--- Final Accountant Handoff Report ---")
        print(json.dumps(report, indent=4))
        return report

# Execution
if __name__ == "__main__":
    messy_whatsapp_message = "Hey, bought parts from Rao Manufacturing today. Base is 45000 plus 8100 GST. Total 53100. Invoice 1042. Need this filed soon."
    agent = LedgerLinkAgent()
    agent.prepare_accountant_handoff(messy_whatsapp_message)

/tmp/ipykernel_8914/1025758161.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


--- 1. Agent Extracting Messy Input ---
--- 2. Python Engine Verifying Compliance Guardrails ---

--- Final Accountant Handoff Report ---
{
    "status": "Ready for Accountant Review",
    "date_processed": "2026-11-11",
    "structured_ledger_entry": {
        "invoice_number": "1042",
        "supplier_name": "Rao Manufacturing",
        "base_amount": 45000.0,
        "gst_amount": 8100.0,
        "total_amount": 53100.0
    },
    "compliance_flags": [
        "URGENT: Total exceeds \u20b950,000. E-way bill generation required."
    ]
}
